# 12장 경량화와 배포

이 장의 예제와 실습을 실행하는 노트북입니다. 개념 설명은 HTML 교재 `12_경량화와_배포.html`에 있습니다. 위에서부터 차례로 실행합니다. 실습의 풀이는 교재의 실습 상자에 접혀 있습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(공개 LLM, 임베딩 모델, 리랭커)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |
| `EMBED_MODEL` | 설정하지 않아도 됩니다. 다른 임베딩 모델을 쓰거나 모델 파일이 든 폴더를 따로 받았을 때, 모델 이름이나 폴더 경로를 넣습니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install transformers torch
# 12장의 예제는 1장의 공개 모델 Qwen2.5-0.5B-Instruct를 PC의 CPU에서 실행합니다. 처음 한 번 약 1GB를 내려받습니다
# 예제 12-1~12-3과 실습 12-1은 API 키 없이 실행할 수 있습니다. 실습 12-2만 API를 호출합니다
import csv

import embed
import torch

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
embed.ask_endpoint(MODEL)                         # 모델이 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as transformers_logging
transformers_logging.disable_progress_bar()
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)   # 파라미터마다 4바이트(fp32)로 올립니다
model.generation_config.update(do_sample=False, temperature=None, top_p=None, top_k=None)  # 언제나 1위 토큰을 고릅니다

with open(f'{DATA}/후공정_문장.txt', encoding='utf-8') as file:
    sentences = [line.strip() for line in file if line.strip()][:10]   # 손실을 계산할 후공정 문장 10개
with open(f'{DATA}/알람.csv', encoding='utf-8') as file:
    alarms = list(csv.DictReader(file))           # 11장의 설비 알람 96건. 열: 번호, 알람, 표준, 용도


def answer(model, question, max_new_tokens=30):
    """질문 하나에 대한 모델의 답"""
    messages = [{'role': 'system', 'content': '질문에 한 문장으로 답한다.'}, {'role': 'user', 'content': question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def mean_loss(model):
    """후공정 문장 10개의 평균 손실(2장). 낮을수록 모델이 그 문장을 그럴듯하게 봅니다"""
    total = 0.0
    for sentence in sentences:
        ids = tokenizer(sentence, return_tensors='pt')['input_ids']
        with torch.no_grad():
            total += model(ids, labels=ids).loss.item()
    return total / len(sentences)


# 소형 모델의 100만 토큰당 단가(달러). 5장 "LLM API"의 표, `llm.py`가 쓰는 Anthropic 모델 기준(2026-11 확인)
# 사내 API를 쓰면 사내 안내의 단가로 바꿉니다
PRICES = {'input': 0.10, 'output': 0.50}


print(f'PC에서 실행하는 모델: {MODEL}, 알람 {len(alarms)}건')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 12-1. 파라미터 수로 메모리 계산하기

교재의 절: "파라미터 수와 GPU 메모리"

준비 셀에서 올린 모델의 파라미터 수를 세어 정밀도마다 가중치의 메모리를 계산하고, 실제로 올린 모델의 메모리와 비교합니다. 이어서 모델 설정(`config`)의 층 수와 어텐션 크기로 KV 캐시와 전체 파인튜닝에 드는 메모리를 계산합니다. 계산값과 측정값이 맞는지 봅니다.

In [ ]:
params = sum(p.numel() for p in model.parameters())
print(f'파라미터 {params:,}개')
for precision, size in [('fp32', 4), ('bf16', 2), ('int8', 1), ('int4', 0.5)]:   # 파라미터 하나의 바이트 수
    print(f'  {precision}: 가중치 {params * size / 1e9:.2f}GB')
print(f'측정: fp32로 올린 모델 {model.get_memory_footprint() / 1e9:.2f}GB')

config = model.config
head_dim = config.hidden_size // config.num_attention_heads
# 층마다 키와 값(2개) × KV 헤드 수 × 헤드 크기 × bf16 2바이트
per_token = 2 * config.num_hidden_layers * config.num_key_value_heads * head_dim * 2
print(f'KV 캐시: 토큰 하나에 {per_token:,}바이트, 32,768토큰이면 {per_token * 32768 / 1e9:.2f}GB')
print(f'전체 파인튜닝(파라미터마다 16바이트): {params * 16 / 1e9:.1f}GB + 활성화값')

파라미터 494,032,768개
  fp32: 가중치 1.98GB
  bf16: 가중치 0.99GB
  int8: 가중치 0.49GB
  int4: 가중치 0.25GB
측정: fp32로 올린 모델 1.98GB
KV 캐시: 토큰 하나에 12,288바이트, 32,768토큰이면 0.40GB
전체 파인튜닝(파라미터마다 16바이트): 7.9GB + 활성화값


## 예제 12-2. 행렬 하나를 양자화하기

교재의 절: "양자화"

0번 층 어텐션의 `q_proj` 행렬을 int8과 int4로 양자화했다가 되돌려, 원래 값과의 평균 오차와 0으로 반올림된 값의 비율을 계산합니다. 스케일을 행렬 전체에 하나만 둘 때와 64개 블록마다 둘 때를 비교합니다. 어느 조합에서 오차가 크게 늘어나는지 봅니다.

In [ ]:
def quantize(weight, bits, block_size=None):
    """weight를 bits비트 정수 단계로 반올림했다가 실수로 되돌립니다. block_size가 있으면 블록마다 스케일을 따로 둡니다"""
    levels = 2 ** (bits - 1) - 1                            # int8은 127, int4는 7
    groups = weight.reshape(-1, block_size) if block_size else weight.reshape(1, -1)
    scale = groups.abs().amax(dim=1, keepdim=True) / levels  # 그룹에서 가장 큰 절댓값이 가장 큰 정수가 됩니다
    return (torch.round(groups / scale) * scale).reshape(weight.shape)


weight = model.model.layers[0].self_attn.q_proj.weight.detach()
print(f'q_proj 값 {weight.numel():,}개: 절댓값 평균 {weight.abs().mean():.4f}, 최댓값 {weight.abs().max():.4f}')
for bits, block_size in [(8, None), (8, 64), (4, None), (4, 64)]:
    restored = quantize(weight, bits, block_size)
    error = ((restored - weight).abs().mean() / weight.abs().mean()).item()   # 절댓값 평균에 대한 평균 오차
    zeros = ((restored == 0) & (weight != 0)).float().mean().item()
    where = '64개 블록마다' if block_size else '행렬 전체에 하나'
    print(f'int{bits}, 스케일 {where}: 평균 오차 {error:.1%}, 0이 된 값 {zeros:.0%}')

q_proj 값 802,816개: 절댓값 평균 0.0420, 최댓값 1.2266
int8, 스케일 행렬 전체에 하나: 평균 오차 5.7%, 0이 된 값 11%
int8, 스케일 64개 블록마다: 평균 오차 0.7%, 0이 된 값 1%
int4, 스케일 행렬 전체에 하나: 평균 오차 74.1%, 0이 된 값 89%
int4, 스케일 64개 블록마다: 평균 오차 13.5%, 0이 된 값 18%


## 예제 12-3. 양자화한 모델의 손실과 답

교재의 절: "양자화"

모델의 모든 층에 있는 행렬을 예제 12-2의 `quantize()`로 양자화한 사본을 만들고, 후공정 문장 10개의 평균 손실과 질문 하나의 답을 원래 모델과 비교합니다. 이 예제는 반올림 오차만 흉내 내므로 메모리는 줄지 않습니다. CPU에서 1~2분 걸립니다. 손실이 크게 오르는 조합에서 답이 어떻게 되는지 봅니다.

In [ ]:
import copy

question = '물이 끓는 온도는 섭씨 몇 도인가요?'
for bits, block_size in [(None, None), (8, None), (4, None), (4, 64)]:
    quantized = copy.deepcopy(model)
    if bits:
        for name, layer in quantized.named_modules():
            if isinstance(layer, torch.nn.Linear) and name != 'lm_head':   # 층의 행렬만 양자화합니다
                layer.weight.data = quantize(layer.weight.data, bits, block_size)
    label = '원래 모델(fp32)' if bits is None else f"int{bits}, {'64개 블록마다' if block_size else '행렬 전체에 하나'}"
    print(f'{label}: 평균 손실 {mean_loss(quantized):.3f}, 답: {answer(quantized, question)}')
    del quantized

원래 모델(fp32): 평균 손실 4.148, 답: 물이 끓는 온도는 섭씨 100도입니다.
int8, 행렬 전체에 하나: 평균 손실 4.345, 답: 물이 끓는 온도는 섭씨 100도입니다.
int4, 행렬 전체에 하나: 평균 손실 17.300, 답: ,retainiguienteeraisörtimir //{
estro (){ }}"><alker()?>FromBodyois jScrollPanekeiteniais快报();?>.TryParseektaturasstasynoinspectionasje negó🏽grily removeFromSuperviewzej Renders
int4, 64개 블록마다: 평균 손실 4.655, 답: 물이 냈다면 섭씨 100도입니다.


## 실습 12-1. 학습 방법별 GPU 메모리

교재의 절: "개념 확인 실습"

파라미터 수와 학습 방법(`full`, `lora`, `qlora`)을 받아 학습에 드는 메모리(GB, 활성화값 제외)를 돌려주는 `training_memory_gb()`를 작성합니다. 전체 파인튜닝은 파라미터마다 16바이트, LoRA는 bf16 가중치 2바이트, QLoRA는 4비트 가중치 0.5바이트로 계산하고, LoRA와 QLoRA는 LoRA 행렬(파라미터의 0.5%)에 파라미터마다 16바이트를 더합니다. 활성화값을 위한 여유가 남아야 들어가는 것으로 보고, 1.5B, 7B, 14B, 70B 모델이 방법마다 24GB GPU 한 장에 들어갈지 먼저 `my_guess`에 적습니다. 결과를 보고, 70B 모델을 QLoRA로 학습하려면 GPU 메모리가 얼마나 필요한지 설명하면 끝납니다.

In [ ]:
def training_memory_gb(params, method, lora_ratio=0.005):
    """학습에 드는 메모리(GB, 활성화값 제외). method는 'full', 'lora', 'qlora'"""
    return 0.0  # TODO


my_guess = {'1.5B': None, '7B': None, '14B': None, '70B': None}  # TODO: 24GB에 들어가는 방법을 적습니다
for name, params in [('1.5B', 1.5e9), ('7B', 7e9), ('14B', 14e9), ('70B', 70e9)]:
    row = ', '.join(f'{method} {training_memory_gb(params, method):.1f}GB' for method in ['full', 'lora', 'qlora'])
    print(f'{name}: {row} (예측 {my_guess[name]})')

## 실습 12-2. 상용 API와 GPU의 손익분기점

교재의 절: "개념 확인 실습"

11장의 알람 다섯 건을 규칙을 적은 시스템 프롬프트(`RULES`)와 함께 소형 모델에 보내, 요청 하나의 평균 입력 토큰과 출력 토큰을 측정합니다. 5장의 소형 단가(`PRICES`)로 요청 하나의 요금을 계산하고, GPU 한 장의 하루 비용 `GPU_COST_PER_DAY`와 같아지는 하루 요청 수를 `break_even()`으로 계산합니다. `GPU_COST_PER_DAY`는 계산을 위해 예로 든 값이므로 사내 기준으로 바꿔 넣습니다. 손익분기점을 먼저 `my_guess`에 어림해 적습니다. 결과를 보고, 하루 1,000건인 업무와 하루 10만 건인 업무에 각각 무엇을 검토할지 설명하면 끝납니다. 이 셀은 API를 5번 호출합니다.

In [ ]:
RULES = ('설비 알람을 사내 표준 형식 "설비코드 | 분류 | 담당"으로 바꿔 답한다. 다른 말은 쓰지 않는다.\n'
         '설비코드: 다이싱 소 DS, 다이 본더 DB, 와이어 본더 WB, 몰딩기 MD, 마킹기 MK, 테스트 핸들러 TH 뒤에 '
         '호기 두 자리를 붙인다. 예: 3호기 와이어 본더 → WB-03\n'
         '분류: 사람이 다치거나 다칠 위험이 있으면 안전, 제품의 측정값이나 외관이 기준을 벗어났으면 품질, '
         '그 밖의 설비 동작 이상이면 설비\n'
         '담당: 설비는 보전팀, 품질은 품질팀, 안전은 안전팀')
GPU_COST_PER_DAY = 20.0   # 달러. 계산을 위해 예로 든 값입니다. 사내 기준으로 바꿉니다


def break_even(cost_per_request, gpu_cost_per_day=GPU_COST_PER_DAY):
    """API의 하루 요금이 GPU의 하루 비용과 같아지는 하루 요청 수"""
    return 0  # TODO


my_guess = None  # TODO: 하루 몇 건에서 같아질지 어림해 적습니다
replies = [llm.ask(row['알람'], system=RULES, model='소형', max_tokens=2000) for row in alarms[:5]]
for row, reply in zip(alarms[:5], replies):
    print(f"{row['알람']} → {reply.text.strip()}  (정답 {row['표준']})")
input_tokens = sum(reply.input_tokens for reply in replies) / len(replies)
output_tokens = sum(reply.output_tokens for reply in replies) / len(replies)
cost_per_request = (input_tokens * PRICES['input'] + output_tokens * PRICES['output']) / 1_000_000
print(f'\n요청 하나: 입력 {input_tokens:.0f}토큰, 출력 {output_tokens:.0f}토큰, {cost_per_request:.6f}달러')
print(f'손익분기점: 하루 {break_even(cost_per_request):,.0f}건 (예측 {my_guess})')